In [42]:
!pip install retina-face

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [43]:
!pip install deepface

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [44]:
from retinaface import RetinaFace # for face detection
from deepface import DeepFace # for face embedding and recognition
import cv2
import numpy as np
import matplotlib.pyplot as plt

In [45]:
def show_img(img, t):
  plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
  plt.title(t)
  plt.show()

In [46]:
def show_two_imgs(img1, img2, t1="org", t2="edited"):

  plt.subplot(121)
  plt.imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
  plt.title(t1)

  plt.subplot(122)
  plt.imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
  plt.title(t2)
  plt.show()

In [47]:
def face_detection(img_path):
  faces = RetinaFace.detect_faces(img_path)
  if not faces:
    raise ValueError("No face detected in the image")
  valid_faces = [f for f in faces.values()]
  if len(valid_faces) > 1:
    raise ValueError("More than one face detected")
  return valid_faces[0]

In [48]:
def is_face_straight(face, max_angle=10):
    landmarks = face["landmarks"]
    left_eye, right_eye = landmarks["left_eye"], landmarks["right_eye"]
    dy = left_eye[1] - right_eye[1]
    dx = left_eye[0] - right_eye[0]
    angle = abs(np.degrees(np.arctan2(dy, dx)))
    return angle <= max_angle

In [49]:
def crop_face_with_margin(img, face, margin=0.2732):  # 0.2732
    h, w = img.shape[:2]
    x1, y1, x2, y2 = face["facial_area"]

    bw, bh = x2 - x1, y2 - y1
    mx, my = int(bw * margin), int(bh * margin)

    x1 = max(0, x1 - mx)
    y1 = max(0, y1 - my)
    x2 = min(w, x2 + mx)
    y2 = min(h, y2 + my)

    return img[y1:y2, x1:x2]


In [50]:
def face_alignment(img, face):
  x1, y1, x2, y2 = face["facial_area"]
  landmarks = face["landmarks"]

  left_eye = landmarks["left_eye"]
  right_eye = landmarks["right_eye"]

  dy = left_eye[1] - right_eye[1]
  dx = left_eye[0] - right_eye[0]

  angle = np.degrees(np.arctan2(dy, dx))
  (h, w) = img.shape[:2]
  eye_center = (
    int((left_eye[0] + right_eye[0]) / 2),
    int((left_eye[1] + right_eye[1]) / 2)
  )

  M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
  aligned_img = cv2.warpAffine(img, M, (w, h))
  return aligned_img

In [51]:
def img_preprocessing(img_path):
  img = cv2.imread(img_path)
  if img is None:
    raise ValueError("Failed to read image")

  face = face_detection(img)

  aligned_img = face_alignment(img, face)
  face = face_detection(aligned_img)
  face_crop = crop_face_with_margin(aligned_img, face, 0.5)

  if face_crop.size == 0:
    raise ValueError("Face crop failed")

  face_img = cv2.cvtColor(face_crop, cv2.COLOR_BGR2GRAY)
  face_img = cv2.cvtColor(face_img, cv2.COLOR_GRAY2RGB)

  return face_img


Face embedding and matching

In [52]:
def extract_embedding(img_path):
    x = img_preprocessing(img_path)
    embedding_obj = DeepFace.represent(
                img_path=x,
                model_name= "ArcFace",
                detector_backend= "retinaface",
                enforce_detection=True
            )
    embedding = embedding_obj[0]["embedding"]
    return  embedding / np.linalg.norm(embedding)

In [53]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [54]:
def is_same_person(img1_path, img2_path, threshold= 0.45):
    img1 = extract_embedding(img1_path)
    img2 = extract_embedding(img2_path)
    sim = cosine_similarity(img1, img2)
    
    return sim >= threshold

In [55]:
img1 = "p13.jpg"
img2 = "p12.jpg"
print(is_same_person(img1, img2))

True
